# Terrestrial Hazard Definitions: V2 Data and Known-Event Checks

This standalone notebook inventories where validation logic currently lives, then provides a bounded workspace for comparing terrestrial-weather V2 fields and candidate hazard definitions with documented events. It does **not** edit or depend on the MYRIAD pipeline notebook's in-memory state.

## Existing validation examined in the repository

- The MYRIAD pipeline notebook contains optional case-window and threshold/mask diagnostics, plus event attrition and linkage diagnostics. Its named case windows are Billings winter cold (Jan 10–16), California precipitation (Feb 3–8), Illinois convection (Mar 13–15), Del Rio heat (May 25–30), Northeast Maine heat (Jun 17–21), and Park Fire onset (Jul 22–29, 2024). These are review candidates, not all independently validated events. A stored run of the optional case-window cell failed before completing the loop.
- The pipeline also contains stored results for a California atmospheric river (Feb 4–7; precipitation expected, stored detection peak Feb 5 04:00 UTC) and Southern Illinois severe convection (Mar 14; precipitation and wind expected). These came from the legacy product; the source cell has a syntax/statement-concatenation issue, and no independent citation is attached. Treat them as leads, not verified labels.
- The May Gannon-storm notebooks and Eagle-I/GeoE matched analysis are exploratory event studies; they are not validation evidence for terrestrial-weather thresholds.
- V2 has a data dictionary with units, temporal support/statistics, valid-pixel fractions, and NaN-versus-zero semantics. Presently staged V2 data cover May 1–September 1, 2024 (111 daily files), so several earlier case windows cannot yet be tested with V2.
- A one-day May 10 old/V2 comparison found close `t2m` agreement but material differences in QPE and gust; it is a processing diagnostic, not validation against truth. MESH is present in V2 where old `MAXSIZE` was missing.

The notebook will preserve case provenance and distinguish **candidate cases** from **verified labels**. Precision/recall are only computed for references explicitly marked verified with compatible hazard, interval, and geography. A threshold crossing is a candidate detection, not a confirmed physical event.

In [39]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
PIPELINE_NOTEBOOK = NOTEBOOK_DIR / "myriad_algorithm_multihazard_identification.ipynb"
V2_DIR = Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/terrestrial_weather/V2")
OLD_WEATHER_DIR = V2_DIR.parent
EVENT_ARTIFACT_DIR = NOTEBOOK_DIR / "outputs" / "terrestrial_v2_validation"
DATA_DICTIONARY = V2_DIR / "DATA_DICTIONARY.md"

# First pass: a single matched day is inexpensive and is enabled. Multi-day data
# loading and candidate event detection remain opt-in pending reference labels.
CASE_NAMES = ["California atmospheric river", "Southern Illinois severe convection",
              "Gannon storm candidate", "Northeast Maine heat", "Park Fire onset"]
PLOT_MAX_HOURS = 24 * 12
RUN_LOAD_CASES = False
RUN_OLD_V2_COMPARISON = True
RUN_DEFINITION_EXPERIMENTS = False
RUN_EVENT_COMPARISON = False

print({
    "repo_root": str(REPO_ROOT),
    "pipeline_notebook_exists": PIPELINE_NOTEBOOK.exists(),
    "v2_directory_exists": V2_DIR.exists(),
    "v2_data_dictionary_exists": DATA_DICTIONARY.exists(),
    "event_artifact_directory": str(EVENT_ARTIFACT_DIR),
    "case_loading_enabled": RUN_LOAD_CASES,
    "old_v2_one_day_comparison_enabled": RUN_OLD_V2_COMPARISON,
})

{'repo_root': '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience', 'pipeline_notebook_exists': True, 'v2_directory_exists': True, 'v2_data_dictionary_exists': True, 'event_artifact_directory': '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation', 'case_loading_enabled': False, 'old_v2_one_day_comparison_enabled': True}


## 2. Scan repository for existing validation checks

This read-only scan indexes validation-like code and rule definitions. It is a locator, not a judgment that a check is scientifically valid. Notebook cells are read from JSON; they are not executed.

In [40]:
VALIDATION_PATTERNS = {
    "assertion": re.compile(r"\bassert\b|pytest\.raises|assert_frame_equal", re.I),
    "validation_or_quality": re.compile(r"\bvalidat\w*\b|\bquality\b|\bsanity\s*check\b|\bQA\b", re.I),
    "schema_or_coverage": re.compile(r"schema|data_vars|columns|missing.*(field|column|time)|coverage|valid_frac", re.I),
    "threshold_or_rule": re.compile(r"threshold|percentile|quantile|persistence|minimum.*(duration|cell)|rule", re.I),
}


def iter_source_documents(root: Path):
    excluded = {".git", ".venv", "venv", "env", ".tox", "__pycache__", ".ipynb_checkpoints",
                ".mypy_cache", ".pytest_cache", "outputs", "_cache", "site-packages",
                "node_modules", "build", "dist"}
    for path in root.rglob("*"):
        if not path.is_file() or any(part in excluded for part in path.parts):
            continue
        if path.suffix == ".py":
            try:
                yield path, [(f"line {i}", line) for i, line in enumerate(path.read_text(errors="replace").splitlines(), 1)]
            except OSError:
                continue
        elif path.suffix == ".ipynb":
            try:
                notebook = json.loads(path.read_text())
            except (OSError, json.JSONDecodeError):
                continue
            located_lines = []
            for cell_index, cell in enumerate(notebook.get("cells", []), start=1):
                source = cell.get("source", [])
                text = "".join(source) if isinstance(source, list) else str(source)
                located_lines.extend((f"cell {cell_index}, line {line_number}", line)
                                     for line_number, line in enumerate(text.splitlines(), start=1))
            yield path, located_lines


validation_rows = []
for source_path, located_lines in iter_source_documents(REPO_ROOT):
    for locator, line in located_lines:
        for match_type, pattern in VALIDATION_PATTERNS.items():
            if pattern.search(line):
                validation_rows.append({
                    "file_path": source_path.relative_to(REPO_ROOT).as_posix(),
                    "source_locator": locator,
                    "match_type": match_type,
                    "code_snippet": line.strip()[:260],
                })

validation_inventory = pd.DataFrame(validation_rows).drop_duplicates()
if not validation_inventory.empty:
    validation_inventory = validation_inventory.sort_values(
        ["file_path", "match_type", "source_locator"]
    ).reset_index(drop=True)
print("Validation-like source lines:", len(validation_inventory))
display(validation_inventory.groupby(["file_path", "match_type"]).size().unstack(fill_value=0).head(30))
display(validation_inventory[validation_inventory.file_path.str.contains("myriad_algorithm_multihazard_identification", regex=False)].head(30))

Validation-like source lines: 2424


match_type,assertion,schema_or_coverage,threshold_or_rule,validation_or_quality
file_path,,,,
data_processing/aggregate_swdi.py,0,13,0,0
data_processing/check_impact_space_weather.py,0,6,3,1
data_processing/check_july_geoe_source.py,0,1,0,2
data_processing/eaglei_grid_mapping.py,0,2,3,1
data_processing/eaglei_grid_pipeline.py,11,14,1,24
data_processing/geoe_eaglei_windows.py,0,16,17,2
data_processing/get_aggregate_mrms_data.py,0,1,0,0
data_processing/get_aggregate_rtma_data.py,0,1,0,0
data_processing/omni_data.py,0,32,1,2


,file_path,source_locator,match_type,code_snippet
1689,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 21",assertion,"assert np.isfinite(_toy_cutoff_gridded[""Ex""].isel(time=0, y=0, x=0))"
1690,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 22",assertion,"assert np.isfinite(_toy_cutoff_gridded[""Ex""].isel(time=0, y=0, x=1))"
1691,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 23",assertion,"assert np.isnan(_toy_cutoff_gridded[""Ex""].isel(time=0, y=1, x=0))"
1692,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 24",assertion,"assert np.isnan(_toy_cutoff_gridded[""Ex""].isel(time=0, y=1, x=1))"
1693,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 25",assertion,"assert ""space_weather_nearest_source_distance_km"" in _toy_cutoff_gridded"
1694,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 7, line 26",assertion,"assert ""space_weather_nearest_fallback_support"" in _toy_cutoff_gridded"
1695,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 8, line 21",assertion,"assert _toy_gridded.attrs[""gridding_method""].startswith(""linear_delaunay_interpolation"")"
1696,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 8, line 22",assertion,"assert ""space_weather_linear_interpolation_support"" in _toy_gridded"
1697,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 8, line 23",assertion,"assert float(np.nanmax(np.abs(_toy_gridded[""Ex""].values))) < SPACE_WEATHER_FILL_ABS_THRESHOLD"
1698,notebooks/myriad_algorithm_multihazard_identification.ipynb,"cell 8, line 24",assertion,"assert np.isnan(_mask_space_weather_fill_values(np.array([_fill], dtype=np.float32))[0])"


## 3. Locate validation usage in the MH database pipeline notebook

The scan should distinguish production event construction from optional QA cells and stored outputs. This section reports source order and execution counts without running the pipeline.

In [16]:
pipeline_validation_inventory = pd.DataFrame(pipeline_validation_rows)
print("Pipeline notebook found:", pipeline_notebook is not None)
print("Validation/context cells:", len(pipeline_validation_inventory))
if not pipeline_validation_inventory.empty:
    pd.set_option("display.max_colwidth", 180)
    display(pipeline_validation_inventory[["cell_index", "execution_count", "cell_type", "match_count",
                                          "validation_lines", "neighboring_pipeline_steps"]].head(30))
else:
    print("No matching cells found")

Pipeline notebook found: True
Validation/context cells: 44


,cell_index,execution_count,cell_type,match_count,validation_lines,neighboring_pipeline_steps
0,1,NaN,markdown,1,"L10: 4. Run the cells under **Optional Development Diagnostics** only when validating event objects, source availability, spatial footprints, group structure, or rendering case...",cell 1: # MYRIAD-Style CONUS Multi-Hazard Identification | This notebook builds single-hazard events and linked multi-hazard event sets for a selected CONUS analysis period. | ...
1,2,5.0,code,5,"L42: OUTPUT_DIR = OUTPUT_ROOT / ""diagnostics"" / f""myriad_conus_{RUN_LABEL}""\nL58: ""extreme_heat"": {""tail"": ""high"", ""percentile"": 95, ""persistence_steps"": 3, ""threshold_referenc...",cell 1: # MYRIAD-Style CONUS Multi-Hazard Identification | This notebook builds single-hazard events and linked multi-hazard event sets for a selected CONUS analysis period. | ...
2,3,6.0,code,1,"L94: def _quantile_threshold(da: xr.DataArray, q: float) -> xr.DataArray:",cell 2: # MYRIAD-style multi-hazard identification for CONUS | # Pipeline configuration: edit only the analysis window and rules in this cell. | from __future__ import annotati...
3,5,8.0,code,4,"L6: # to the analysis grid, and mask IDL/NetCDF fill values before magnitude or thresholds are computed.\nL10: SPACE_WEATHER_FILL_ABS_THRESHOLD = 1e20\nL33: invalid = ~np.isfin...",cell 4: # GeoE processing override dependencies | import numpy as np | import pandas as pd\ncell 5: # GeoE processing fixes: fill-value masking and interpolation-based regriddi...
4,6,10.0,code,1,"L140: ""fill_value_policy"": f""masked nonfinite values and abs(value) > {SPACE_WEATHER_FILL_ABS_THRESHOLD:.1e}"",",cell 5: # GeoE processing fixes: fill-value masking and interpolation-based regridding | # | # This overrides `_grid_space_weather_to_analysis_grid` from the utility cell above...
5,7,11.0,code,8,"L1: # GeoE no-extrapolation cutoff smoke test\nL21: assert np.isfinite(_toy_cutoff_gridded[""Ex""].isel(time=0, y=0, x=0))\nL22: assert np.isfinite(_toy_cutoff_gridded[""Ex""].isel...","cell 6: # GeoE no-extrapolation cutoff: unsupported cells become NaN | # | # Linear interpolation is retained inside the modeled geoE point cloud. Outside that support,\ncell 7..."
6,8,9.0,code,6,"L1: # GeoE mapper smoke test on a synthetic unstructured grid\nL21: assert _toy_gridded.attrs[""gridding_method""].startswith(""linear_delaunay_interpolation"")\nL22: assert ""space...",cell 7: # GeoE no-extrapolation cutoff smoke test | _toy_space_cutoff = xr.Dataset( | data_vars={\ncell 8: # GeoE mapper smoke test on a synthetic unstructured grid | _fill = n...
7,9,NaN,markdown,2,"L1: ## Optional Development Diagnostics\nL3: The following cells are not required for routine production generation. They validate source coverage, thresholds, event objects, s...",cell 8: # GeoE mapper smoke test on a synthetic unstructured grid | _fill = np.float32(9.96921e36) | _toy_space = xr.Dataset(\ncell 9: ## Optional Development Diagnostics | The...
8,10,NaN,code,6,"L6: AUDIT_DIR = OUTPUT_ROOT / ""diagnostics"" / ""input_audit""\nL78: coverage_year_rows = []\nL83: coverage_year_rows.append({""source"": name, ""year"": year, ""expected_days"": len(ex...","cell 9: ## Optional Development Diagnostics | The following cells are not required for routine production generation. They validate source coverage, thresholds, event objects, ..."
9,11,NaN,code,2,"L5: BENCHMARK_DIR = OUTPUT_ROOT / ""diagnostics"" / ""runtime_benchmark""\nL29: _quantile_threshold(temp_bench, 95).compute()","cell 10: # Full-period input audit | AUDIT_START = pd.Timestamp(""2024-01-01"") | AUDIT_END = pd.Timestamp(""2025-12-31"")\ncell 11: # Runtime benchmark: terrestrial file loading +..."


## 4. Extract validation cells and build an inventory

The inventory below labels what the code appears to check and explicitly marks expected outcomes as unverified where the source only contains a diagnostic or an unexecuted cell. It is an index for review, not a certification report.

In [5]:
def infer_validation_name(text: str) -> str:
    lower = text.lower()
    if "event_windows" in lower or "window_diagnostic" in lower:
        return "known-case window diagnostics"
    if "attrition" in lower:
        return "event-filter attrition"
    if "linkage" in lower or "component" in lower:
        return "pair/group linkage diagnostics"
    if "threshold" in lower or "mask" in lower:
        return "hazard threshold/mask diagnostics"
    if "assert" in lower or "smoke test" in lower:
        return "synthetic smoke/assertion checks"
    return "validation/context code review"


def infer_input_dataset(text: str) -> str:
    names = []
    for token, label in [("ds_terr", "terrestrial weather"), ("ds_space", "NOAA GeoE"),
                         ("eagle", "Eagle-I"), ("wildfire", "wildfire"),
                         ("v2", "terrestrial V2")]:
        if token.lower() in text.lower():
            names.append(label)
    return ", ".join(dict.fromkeys(names)) or "not inferred from matched source lines"


inventory_rows = []
for row in pipeline_validation_rows:
    text = row["validation_lines"]
    inventory_rows.append({
        "validation_name": infer_validation_name(text),
        "input_dataset": infer_input_dataset(text + " " + row["neighboring_pipeline_steps"]),
        "rule_definition": text[:1200],
        "expected_output": "Inspect cell output / stored result; not inferred as passing from source alone.",
        "pipeline_location": f"{PIPELINE_NOTEBOOK.relative_to(REPO_ROOT).as_posix()} — cell {row['cell_index']}",
        "execution_count_in_saved_notebook": row["execution_count"],
        "evidence_status": "source located; execution/result requires review",
    })

validation_table = pd.DataFrame(inventory_rows)
display(validation_table.head(40))

,validation_name,input_dataset,rule_definition,expected_output,pipeline_location,execution_count_in_saved_notebook,evidence_status
0,validation/context code review,not inferred from matched source lines,L10: 4. Run the cells under **Optional Develop...,Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,NaN,source located; execution/result requires review
1,hazard threshold/mask diagnostics,not inferred from matched source lines,"L42: OUTPUT_DIR = OUTPUT_ROOT / ""diagnostics"" ...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,5.0,source located; execution/result requires review
2,hazard threshold/mask diagnostics,not inferred from matched source lines,"L94: def _quantile_threshold(da: xr.DataArray,...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,6.0,source located; execution/result requires review
3,hazard threshold/mask diagnostics,not inferred from matched source lines,"L6: # to the analysis grid, and mask IDL/NetCD...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,8.0,source located; execution/result requires review
4,hazard threshold/mask diagnostics,not inferred from matched source lines,"L140: ""fill_value_policy"": f""masked nonfinite ...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,10.0,source located; execution/result requires review
5,synthetic smoke/assertion checks,not inferred from matched source lines,L1: # GeoE no-extrapolation cutoff smoke test\...,Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,11.0,source located; execution/result requires review
6,hazard threshold/mask diagnostics,not inferred from matched source lines,L1: # GeoE mapper smoke test on a synthetic un...,Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,9.0,source located; execution/result requires review
7,hazard threshold/mask diagnostics,not inferred from matched source lines,L1: ## Optional Development Diagnostics\nL3: T...,Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,NaN,source located; execution/result requires review
8,validation/context code review,not inferred from matched source lines,"L6: AUDIT_DIR = OUTPUT_ROOT / ""diagnostics"" / ...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,NaN,source located; execution/result requires review
9,hazard threshold/mask diagnostics,not inferred from matched source lines,"L5: BENCHMARK_DIR = OUTPUT_ROOT / ""diagnostics...",Inspect cell output / stored result; not infer...,notebooks/myriad_algorithm_multihazard_identif...,NaN,source located; execution/result requires review


## 5. Branch: keep exploration separate from the MH database pipeline

The scan locates validation/case diagnostics in the MYRIAD notebook. Those cells are entangled with the database workflow and its current operational definitions, so this analysis is being kept in a standalone notebook. The branch below is safe to rerun: it never overwrites an existing notebook and recognizes that the current notebook is already the separate exploration workspace.

In [8]:
CURRENT_NOTEBOOK_PATH = NOTEBOOK_DIR / "terrestrial_hazard_v2_validation.ipynb"
validation_is_in_pipeline = any(
    "myriad_algorithm_multihazard_identification.ipynb" in str(row.get("pipeline_location", ""))
    for row in inventory_rows
)
EXPLORATION_NOTEBOOK_PATH = NOTEBOOK_DIR / "terrestrial_hazard_v2_validation.ipynb"


def starter_notebook_document() -> dict[str, Any]:
    sections = [
        "Load bounded V2 terrestrial-weather cases and inspect variable coverage.",
        "Compare V2 with the previous product only on matched dates, variables, and support.",
        "Parameterize physically interpretable precipitation, wind, and temperature candidates.",
        "Compare detections only to verified event references; candidate windows are not truth labels.",
        "Export diagnostics and mismatch tables without changing source data or production thresholds.",
    ]
    cells = [{"cell_type": "markdown", "metadata": {"language": "markdown"},
              "source": ["# V2 Terrestrial Hazard Exploration\n", "\n", *[f"- {item}\n" for item in sections]]}]
    cells.append({"cell_type": "code", "execution_count": None, "outputs": [],
                  "metadata": {"language": "python"},
                  "source": ["# Starter cell: configure paths and select a bounded case before loading data.\n",
                             "RUN_LOAD_CASES = False\n", "RUN_DEFINITION_EXPERIMENTS = False\n"]})
    return {"cells": cells, "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}},
            "nbformat": 4, "nbformat_minor": 5}


if validation_is_in_pipeline:
    if CURRENT_NOTEBOOK_PATH.resolve() == EXPLORATION_NOTEBOOK_PATH.resolve():
        branch_action = "separate exploration notebook already created; continue here"
    elif EXPLORATION_NOTEBOOK_PATH.exists():
        branch_action = "separate exploration notebook already exists; preserve it"
    else:
        EXPLORATION_NOTEBOOK_PATH.write_text(json.dumps(starter_notebook_document(), indent=2))
        branch_action = "created separate exploration notebook scaffold"
else:
    branch_action = "validation not located in MH DB notebook; in-place continuation is possible"

print({"validation_found_in_mh_db_pipeline": validation_is_in_pipeline,
       "branch_action": branch_action,
       "exploration_notebook": str(EXPLORATION_NOTEBOOK_PATH),
       "will_overwrite": False})

{'validation_found_in_mh_db_pipeline': True, 'branch_action': 'separate exploration notebook already created; continue here', 'exploration_notebook': '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/terrestrial_hazard_v2_validation.ipynb', 'will_overwrite': False}


## 6. Load V2 terrestrial hazards and known-event references

Case windows below were found in existing diagnostics. They begin as `candidate_only`: a name and date range are not ground-truth labels. The cell computes V2 file coverage before loading any gridded arrays. A user-supplied event table can be added later with verified source, hazard, start/end, and optional county FIPS.

In [41]:
CASE_REGISTRY = pd.DataFrame([
    {"case_id": "billings_winter_cold_20240110", "case_name": "Billings winter cold", "hazard": "extreme_cold", "start_time": "2024-01-10", "end_time": "2024-01-16 23:00", "validation_status": "candidate_only", "reference": "Add authoritative event/impact source", "county_fips": None},
    {"case_id": "california_precip_20240203", "case_name": "California precipitation", "hazard": "extreme_precip", "start_time": "2024-02-03", "end_time": "2024-02-08 23:00", "validation_status": "candidate_only", "reference": "Add storm report / gauge or flood reference", "county_fips": None},
    {"case_id": "illinois_convection_20240313", "case_name": "Illinois convection", "hazard": "convective_context", "start_time": "2024-03-13", "end_time": "2024-03-15 23:00", "validation_status": "candidate_only", "reference": "Add authoritative storm report", "county_fips": None},
    {"case_id": "gannon_20240510", "case_name": "Gannon storm candidate", "hazard": "multi_hazard_context", "start_time": "2024-05-10", "end_time": "2024-05-20 23:00", "validation_status": "candidate_only", "reference": "Verify storm boundaries and source; V2 currently lacks May 19-20", "county_fips": None},
    {"case_id": "del_rio_heat_20240525", "case_name": "Del Rio late-May heat", "hazard": "extreme_heat", "start_time": "2024-05-25", "end_time": "2024-05-30 23:00", "validation_status": "candidate_only", "reference": "Add NWS/local impact reference", "county_fips": None},
    {"case_id": "maine_heat_20240617", "case_name": "Northeast Maine heat", "hazard": "extreme_heat", "start_time": "2024-06-17", "end_time": "2024-06-21 23:00", "validation_status": "candidate_only", "reference": "Add NWS/local impact reference", "county_fips": None},
    {"case_id": "park_fire_20240722", "case_name": "Park Fire onset", "hazard": "wildfire_weather_context", "start_time": "2024-07-22", "end_time": "2024-07-29 23:00", "validation_status": "candidate_only", "reference": "Fire inventory can anchor fire dates; define weather-response target separately", "county_fips": None},
])
for column in ["start_time", "end_time"]:
    CASE_REGISTRY[column] = pd.to_datetime(CASE_REGISTRY[column], utc=True)

v2_paths = sorted(V2_DIR.glob("weather_CONUS_*.nc")) if V2_DIR.exists() else []
v2_file_dates = pd.DatetimeIndex(pd.to_datetime(
    [path.stem.rsplit("_", 1)[-1] for path in v2_paths], format="%Y%m%d", errors="coerce", utc=True
)).normalize()
CASE_REGISTRY["expected_days"] = CASE_REGISTRY.apply(
    lambda row: len(pd.date_range(row.start_time.normalize(), row.end_time.normalize(), freq="D")), axis=1
)
CASE_REGISTRY["v2_days_present"] = CASE_REGISTRY.apply(
    lambda row: int(pd.date_range(row.start_time.normalize(), row.end_time.normalize(), freq="D").isin(v2_file_dates).sum()), axis=1
)
CASE_REGISTRY["v2_file_coverage"] = CASE_REGISTRY.v2_days_present / CASE_REGISTRY.expected_days
display(CASE_REGISTRY[["case_name", "hazard", "start_time", "end_time", "v2_days_present", "expected_days", "v2_file_coverage", "validation_status", "reference"]])

# Optionally point this to a curated, verified CSV/Parquet/JSON event reference.
KNOWN_EVENTS_FILE = None
REFERENCE_ALIASES = {
    "event_id": ["event_id", "id", "case_id"], "hazard": ["hazard", "hazard_type", "event_type"],
    "start_time": ["start_time", "start", "begin", "onset"], "end_time": ["end_time", "end", "stop", "recovery"],
    "county_fips": ["county_fips", "fips", "geoid"], "validation_status": ["validation_status", "status"],
    "source": ["source", "reference", "citation"],
}


def load_known_events(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)
    elif path.suffix.lower() == ".json":
        frame = pd.read_json(path)
    else:
        raise ValueError(f"Unsupported event reference format: {path.suffix}")
    lookup = {str(column).strip().lower(): column for column in frame.columns}
    rename = {}
    for canonical, aliases in REFERENCE_ALIASES.items():
        found = next((lookup[a.lower()] for a in aliases if a.lower() in lookup), None)
        if found is not None:
            rename[found] = canonical
    frame = frame.rename(columns=rename)
    for required in ["hazard", "start_time", "end_time"]:
        if required not in frame:
            raise ValueError(f"Known-event table is missing required field: {required}")
    if "event_id" not in frame:
        frame["event_id"] = [f"reference_{i:05d}" for i in range(len(frame))]
    if "validation_status" not in frame:
        frame["validation_status"] = "unverified"
    if "source" not in frame:
        frame["source"] = str(path)
    frame["start_time"] = pd.to_datetime(frame.start_time, errors="coerce", utc=True)
    frame["end_time"] = pd.to_datetime(frame.end_time, errors="coerce", utc=True)
    frame["hazard"] = frame.hazard.astype("string").str.lower().str.strip()
    if "county_fips" in frame:
        frame["county_fips"] = frame.county_fips.astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    return frame.dropna(subset=["start_time", "end_time", "hazard"]).copy()


known_events = load_known_events(KNOWN_EVENTS_FILE) if KNOWN_EVENTS_FILE else pd.DataFrame(
    columns=["event_id", "hazard", "start_time", "end_time", "county_fips", "validation_status", "source"]
)
print("Verified reference events available:", int(known_events.validation_status.astype(str).str.lower().isin({"verified", "confirmed"}).sum()) if not known_events.empty else 0)

# Matched-day old-versus-V2 comparison, only on dates with both versions.
OLD_V2_COMPARISON_DATE = "2024-05-10"
COMPARISON_VARIABLES = [
    "MultiSensor_QPE_01H_Pass2_00.00", "i10fg", "wspd10", "t2m", "t2m_max", "t2m_min",
    "NLDN_CG_015min_AvgDensity_00.00", "NLDN_CG_flashes_1h",
    "MESH_Max_60min_00.50", "EchoTop_18_00.50", "MergedReflectivityComposite_00.50",
]
NONNEGATIVE_VARIABLES = {
    "MultiSensor_QPE_01H_Pass2_00.00", "i10fg", "wspd10", "t2m_max", "t2m_min",
    "NLDN_CG_015min_AvgDensity_00.00", "NLDN_CG_flashes_1h",
    "MESH_Max_60min_00.50", "EchoTop_18_00.50",
}


def cleaned_comparison_values(data_array, variable_name):
    values = np.asarray(data_array.values, dtype=float).copy()
    invalid = ~np.isfinite(values) | (np.abs(values) > 1e20)
    for metadata in (data_array.attrs, data_array.encoding):
        for key in ("_FillValue", "missing_value"):
            if key in metadata:
                for fill in np.asarray(metadata[key]).reshape(-1):
                    if np.isfinite(float(fill)):
                        invalid |= values == float(fill)
    # MRMS negative sentinels (e.g. -999 dBZ) are not physical measurements.
    if variable_name in NONNEGATIVE_VARIABLES:
        invalid |= values < 0
    elif "reflectivity" in variable_name.lower():
        invalid |= values <= -900
    values[invalid] = np.nan
    return values


def compare_old_v2_day(date_text, variables=COMPARISON_VARIABLES, min_v2_valid_fraction=0.9):
    filename = f"weather_CONUS_{pd.Timestamp(date_text):%Y%m%d}.nc"
    old_path, new_path = OLD_WEATHER_DIR / filename, V2_DIR / filename
    if not old_path.exists() or not new_path.exists():
        raise FileNotFoundError(f"Matched comparison needs both products: {old_path}; {new_path}")
    rows = []
    with xr.open_dataset(old_path) as old_raw, xr.open_dataset(new_path) as new_raw:
        common = [name for name in variables if name in old_raw and name in new_raw]
        for name in common:
            old, new = xr.align(old_raw[name], new_raw[name], join="inner")
            if old.dims != new.dims or old.shape != new.shape:
                rows.append({"variable": name, "comparison_status": "dimensions differ",
                             "old_shape": str(old.shape), "v2_shape": str(new.shape)})
                continue
            x = cleaned_comparison_values(old, name)
            y = cleaned_comparison_values(new, name)
            valid_x, valid_y = np.isfinite(x), np.isfinite(y)
            paired = valid_x & valid_y
            v2_coverage_name = f"{name}_valid_frac"
            if v2_coverage_name in new_raw:
                coverage, _ = xr.align(new_raw[v2_coverage_name], new, join="inner")
                support = np.asarray(coverage.values, dtype=float) >= min_v2_valid_fraction
            else:
                support = np.ones(y.shape, dtype=bool)
            paired_supported = paired & support
            xv, yv = x[paired_supported], y[paired_supported]
            rows.append({
                "variable": name, "comparison_status": "matched grid/time points; invalid sentinels masked",
                "old_units": old.attrs.get("units", "not stored in old file"), "v2_units": new.attrs.get("units"),
                "n_total": int(x.size), "old_valid_fraction": float(valid_x.mean()),
                "v2_valid_fraction": float(valid_y.mean()), "paired_valid_fraction": float(paired.mean()),
                "paired_v2_supported_fraction": float(paired_supported.mean()),
                "v2_support_cutoff": min_v2_valid_fraction if v2_coverage_name in new_raw else np.nan,
                "old_min": float(np.min(xv)) if len(xv) else np.nan,
                "old_max": float(np.max(xv)) if len(xv) else np.nan,
                "v2_min": float(np.min(yv)) if len(yv) else np.nan,
                "v2_max": float(np.max(yv)) if len(yv) else np.nan,
                "bias_v2_minus_old": float(np.mean(yv - xv)) if len(xv) else np.nan,
                "mae": float(np.mean(np.abs(yv - xv))) if len(xv) else np.nan,
                "rmse": float(np.sqrt(np.mean((yv - xv) ** 2))) if len(xv) else np.nan,
                "correlation": float(np.corrcoef(xv, yv)[0, 1]) if len(xv) > 1 and np.std(xv) and np.std(yv) else np.nan,
                "support_caveat": "Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.",
            })
    return pd.DataFrame(rows)


if RUN_OLD_V2_COMPARISON:
    old_v2_comparison = compare_old_v2_day(OLD_V2_COMPARISON_DATE)
    display(old_v2_comparison)
else:
    old_v2_comparison = pd.DataFrame()
    print("Old/V2 matched-day comparison is opt-in. Default date:", OLD_V2_COMPARISON_DATE)

,case_name,hazard,start_time,end_time,v2_days_present,expected_days,v2_file_coverage,validation_status,reference
0,Billings winter cold,extreme_cold,2024-01-10 00:00:00+00:00,2024-01-16 23:00:00+00:00,0,7,0.000000,candidate_only,Add authoritative event/impact source
1,California precipitation,extreme_precip,2024-02-03 00:00:00+00:00,2024-02-08 23:00:00+00:00,0,6,0.000000,candidate_only,Add storm report / gauge or flood reference
2,Illinois convection,convective_context,2024-03-13 00:00:00+00:00,2024-03-15 23:00:00+00:00,0,3,0.000000,candidate_only,Add authoritative storm report
3,Gannon storm candidate,multi_hazard_context,2024-05-10 00:00:00+00:00,2024-05-20 23:00:00+00:00,9,11,0.818182,candidate_only,Verify storm boundaries and source; V2 currently lacks May 19-20
4,Del Rio late-May heat,extreme_heat,2024-05-25 00:00:00+00:00,2024-05-30 23:00:00+00:00,0,6,0.000000,candidate_only,Add NWS/local impact reference
5,Northeast Maine heat,extreme_heat,2024-06-17 00:00:00+00:00,2024-06-21 23:00:00+00:00,5,5,1.000000,candidate_only,Add NWS/local impact reference
6,Park Fire onset,wildfire_weather_context,2024-07-22 00:00:00+00:00,2024-07-29 23:00:00+00:00,8,8,1.000000,candidate_only,Fire inventory can anchor fire dates; define weather-response target separately


Verified reference events available: 0


,variable,comparison_status,old_units,v2_units,n_total,old_valid_fraction,v2_valid_fraction,paired_valid_fraction,paired_v2_supported_fraction,v2_support_cutoff,old_min,old_max,v2_min,v2_max,bias_v2_minus_old,mae,rmse,correlation,support_caveat
0,MultiSensor_QPE_01H_Pass2_00.00,matched grid/time points; invalid sentinels masked,not stored in old file,mm,137280,1.000000,0.979720,0.979720,0.979720,0.9,0.000000,28.067677,0.000000,89.199997,3.208449e-01,0.346782,2.418600,0.439069,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.
1,i10fg,matched grid/time points; invalid sentinels masked,not stored in old file,m s-1,137280,1.000000,1.000000,1.000000,1.000000,NaN,0.320647,20.863410,0.710000,44.720001,2.136122e+00,2.136122,2.739491,0.882870,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.
2,wspd10,matched grid/time points; invalid sentinels masked,not stored in old file,m s-1,137280,1.000000,1.000000,1.000000,1.000000,NaN,0.000470,16.529590,0.180000,28.590000,1.942067e+00,1.942067,2.430349,0.847758,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.
3,t2m,matched grid/time points; invalid sentinels masked,not stored in old file,K,137280,1.000000,1.000000,1.000000,1.000000,NaN,265.025460,313.157908,265.025452,313.157898,6.260659e-09,0.000008,0.000009,1.000000,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.
4,EchoTop_18_00.50,matched grid/time points; invalid sentinels masked,not stored in old file,km,137280,0.192818,0.902098,0.138971,0.124738,0.9,0.000000,16.475838,0.000000,19.000000,2.949047e+00,2.949047,5.369362,0.664585,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.
5,MergedReflectivityComposite_00.50,matched grid/time points; invalid sentinels masked,not stored in old file,dBZ,137280,0.953438,0.902098,0.899592,0.876537,0.9,-185.792453,48.892093,-30.000000,83.000000,7.524974e+01,76.743121,86.709230,0.300964,Old product has no per-cell source coverage mask; paired zeros may represent prior zero-fill.


### Existing MH database known-event checks found in the pipeline

Beyond the earlier case-window review queue, the MYRIAD notebook has a later `known_event_validation` section with stored results for a California atmospheric river (Feb 4–7, 2024) and Southern Illinois severe convection (Mar 14). Its saved results show precipitation detections in both windows; the Illinois wind expectation is marked true but has zero regional active-cell hours. Those outputs are from the pipeline's prior processed product, not terrestrial V2. The corresponding source cell currently has concatenated statements at its end and should be repaired before rerunning. Neither case has an authoritative citation stored alongside its expected-detection flag, so the results are leads, not verified labels.

In [45]:
PIPELINE_KNOWN_EVENT_RESULTS = pd.DataFrame([
    {"case_id": "pipeline_ca_ar_precip_20240204", "case_name": "California atmospheric river", "hazard": "extreme_precip", "start_time": "2024-02-04", "end_time": "2024-02-07 23:00", "stored_expected_detection": True, "stored_regional_active_cell_hours": 658, "stored_regional_peak_cells": 27, "stored_peak_time_utc": "2024-02-05 04:00", "stored_regional_field_peak": 10.625932, "stored_conus_active_cell_hours": 3330, "stored_regional_share": 0.197598},
    {"case_id": "pipeline_il_precip_20240314", "case_name": "Southern Illinois severe convection", "hazard": "extreme_precip", "start_time": "2024-03-13", "end_time": "2024-03-15 23:00", "stored_expected_detection": True, "stored_regional_active_cell_hours": 267, "stored_regional_peak_cells": 20, "stored_peak_time_utc": "2024-03-15 00:00", "stored_regional_field_peak": 16.734007, "stored_conus_active_cell_hours": 1683, "stored_regional_share": 0.158645},
    {"case_id": "pipeline_il_wind_20240314", "case_name": "Southern Illinois severe convection", "hazard": "extreme_wind", "start_time": "2024-03-13", "end_time": "2024-03-15 23:00", "stored_expected_detection": True, "stored_regional_active_cell_hours": 0, "stored_regional_peak_cells": 0, "stored_peak_time_utc": None, "stored_regional_field_peak": 18.242758, "stored_conus_active_cell_hours": 351, "stored_regional_share": 0.0},
])
for column in ["start_time", "end_time"]:
    PIPELINE_KNOWN_EVENT_RESULTS[column] = pd.to_datetime(PIPELINE_KNOWN_EVENT_RESULTS[column], utc=True)
PIPELINE_KNOWN_EVENT_RESULTS["validation_status"] = "stored pipeline result; reference citation not attached"
PIPELINE_KNOWN_EVENT_RESULTS["v2_days_present"] = PIPELINE_KNOWN_EVENT_RESULTS.apply(
    lambda row: int(pd.date_range(row.start_time.normalize(), row.end_time.normalize(), freq="D").isin(v2_file_dates).sum()), axis=1
)
PIPELINE_KNOWN_EVENT_RESULTS["expected_days"] = PIPELINE_KNOWN_EVENT_RESULTS.apply(
    lambda row: len(pd.date_range(row.start_time.normalize(), row.end_time.normalize(), freq="D")), axis=1
)

# Add only the case/hazard pairs not already in the general registry.
known_case_rows = PIPELINE_KNOWN_EVENT_RESULTS[["case_id", "case_name", "hazard", "start_time", "end_time", "validation_status"]].copy()
known_case_rows["reference"] = "Stored in MH notebook; no independent citation attached"
known_case_rows["county_fips"] = None
known_case_rows["v2_days_present"] = PIPELINE_KNOWN_EVENT_RESULTS.v2_days_present
known_case_rows["expected_days"] = PIPELINE_KNOWN_EVENT_RESULTS.expected_days
known_case_rows["v2_file_coverage"] = known_case_rows.v2_days_present / known_case_rows.expected_days
existing_ids = set(CASE_REGISTRY.case_id)
CASE_REGISTRY = pd.concat([CASE_REGISTRY, known_case_rows.loc[~known_case_rows.case_id.isin(existing_ids)]], ignore_index=True)
print("Saved MH known-event checks (not yet verified ground truth):")
display(PIPELINE_KNOWN_EVENT_RESULTS)

Saved MH known-event checks (not yet verified ground truth):


,case_id,case_name,hazard,start_time,end_time,stored_expected_detection,stored_regional_active_cell_hours,stored_regional_peak_cells,stored_peak_time_utc,stored_regional_field_peak,stored_conus_active_cell_hours,stored_regional_share,validation_status,v2_days_present,expected_days
0,pipeline_ca_ar_precip_20240204,California atmospheric river,extreme_precip,2024-02-04 00:00:00+00:00,2024-02-07 23:00:00+00:00,True,658,27,2024-02-05 04:00,10.625932,3330,0.197598,stored pipeline result; reference citation not attached,0,4
1,pipeline_il_precip_20240314,Southern Illinois severe convection,extreme_precip,2024-03-13 00:00:00+00:00,2024-03-15 23:00:00+00:00,True,267,20,2024-03-15 00:00,16.734007,1683,0.158645,stored pipeline result; reference citation not attached,0,3
2,pipeline_il_wind_20240314,Southern Illinois severe convection,extreme_wind,2024-03-13 00:00:00+00:00,2024-03-15 23:00:00+00:00,True,0,0,None,18.242758,351,0.000000,stored pipeline result; reference citation not attached,0,3


### Interpret old/V2 differences using processing semantics

The legacy MRMS gridding helper used a 50 km cell **mean** and converted empty bins to zero. V2's data dictionary reports MRMS cell **maxima**, NaN for no coverage, and a valid-pixel fraction. RTMA V2 labels each hourly record with the analysis valid at HH:00; do not treat it as an hourly mean. Consequently, raw old/V2 bias, MAE, RMSE and correlation are descriptive of two different processed products, not a pure reprocessing error. The old files lack units and coverage masks. This guarded comparison masks physical-sign violations and obvious reflectivity fill contamination and also reports metrics on V2 cells with at least 90% source-pixel support; inspect source semantics before interpreting differences.

In [43]:
MRMS_VARIABLES = {
    "MultiSensor_QPE_01H_Pass2_00.00", "NLDN_CG_015min_AvgDensity_00.00",
    "NLDN_CG_flashes_1h", "MESH_Max_60min_00.50", "EchoTop_18_00.50",
    "MergedReflectivityComposite_00.50",
}
NONNEGATIVE_VARIABLES = {
    "MultiSensor_QPE_01H_Pass2_00.00", "i10fg", "wspd10", "t2m_max", "t2m_min",
    "NLDN_CG_015min_AvgDensity_00.00", "NLDN_CG_flashes_1h",
    "MESH_Max_60min_00.50", "EchoTop_18_00.50",
}


def guarded_comparison_values(data_array, variable_name):
    values = np.asarray(data_array.values, dtype=float).copy()
    invalid = ~np.isfinite(values) | (np.abs(values) > 1e20)
    for metadata in (data_array.attrs, data_array.encoding):
        for key in ("_FillValue", "missing_value"):
            if key in metadata:
                for fill in np.asarray(metadata[key]).reshape(-1):
                    if np.isfinite(float(fill)):
                        invalid |= values == float(fill)
    if variable_name in NONNEGATIVE_VARIABLES:
        invalid |= values < 0
    elif "reflectivity" in variable_name.lower():
        # Diagnostic mask for impossible legacy aggregate values after negative
        # sentinels may already have contaminated the cell mean.
        invalid |= values <= -100
    values[invalid] = np.nan
    return values


def compare_old_v2_with_support(date_text, variables=COMPARISON_VARIABLES, min_v2_valid_fraction=0.9):
    filename = f"weather_CONUS_{pd.Timestamp(date_text):%Y%m%d}.nc"
    old_path, new_path = OLD_WEATHER_DIR / filename, V2_DIR / filename
    rows = []
    with xr.open_dataset(old_path) as old_raw, xr.open_dataset(new_path) as new_raw:
        for name in [v for v in variables if v in old_raw and v in new_raw]:
            old, new = xr.align(old_raw[name], new_raw[name], join="inner")
            x, y = guarded_comparison_values(old, name), guarded_comparison_values(new, name)
            if x.shape != y.shape:
                rows.append({"variable": name, "comparison_status": "dimensions differ"})
                continue
            pair = np.isfinite(x) & np.isfinite(y)
            coverage_name = f"{name}_valid_frac"
            if coverage_name in new_raw:
                coverage, _ = xr.align(new_raw[coverage_name], new, join="inner")
                supported = np.asarray(coverage.values, dtype=float) >= min_v2_valid_fraction
            else:
                supported = np.ones(y.shape, dtype=bool)
            pair_supported = pair & supported
            xv, yv = x[pair_supported], y[pair_supported]
            rows.append({
                "variable": name, "comparison_status": "guarded matched points",
                "old_units": old.attrs.get("units", "not stored"), "v2_units": new.attrs.get("units"),
                "old_cell_statistic": "50-km mean; no-coverage bins zero-filled" if name in MRMS_VARIABLES else "legacy product; inspect processing",
                "v2_cell_statistic": "see V2 data dictionary",
                "n_total": int(x.size), "old_valid_fraction_after_guards": float(np.isfinite(x).mean()),
                "v2_valid_fraction": float(np.isfinite(y).mean()),
                "v2_supported_fraction": float(supported.mean()),
                "paired_supported_fraction": float(pair_supported.mean()),
                "old_min": float(xv.min()) if len(xv) else np.nan,
                "old_max": float(xv.max()) if len(xv) else np.nan,
                "v2_min": float(yv.min()) if len(yv) else np.nan,
                "v2_max": float(yv.max()) if len(yv) else np.nan,
                "bias_v2_minus_old": float(np.mean(yv - xv)) if len(xv) else np.nan,
                "mae": float(np.mean(np.abs(yv - xv))) if len(xv) else np.nan,
                "rmse": float(np.sqrt(np.mean((yv - xv) ** 2))) if len(xv) else np.nan,
                "correlation": float(np.corrcoef(xv, yv)[0, 1]) if len(xv) > 1 and np.std(xv) and np.std(yv) else np.nan,
                "old_coverage_warning": "legacy zeros may combine true zeros and unavailable-source zero-fill",
            })
    return pd.DataFrame(rows)


if RUN_OLD_V2_COMPARISON:
    old_v2_comparison = compare_old_v2_with_support(OLD_V2_COMPARISON_DATE)
    display(old_v2_comparison)
else:
    old_v2_comparison = pd.DataFrame()
    print("Set RUN_OLD_V2_COMPARISON=True for the one-day matched comparison.")

,variable,comparison_status,old_units,v2_units,old_cell_statistic,v2_cell_statistic,n_total,old_valid_fraction_after_guards,v2_valid_fraction,v2_supported_fraction,paired_supported_fraction,old_min,old_max,v2_min,v2_max,bias_v2_minus_old,mae,rmse,correlation,old_coverage_warning
0,MultiSensor_QPE_01H_Pass2_00.00,guarded matched points,not stored,mm,50-km mean; no-coverage bins zero-filled,see V2 data dictionary,137280,1.000000,0.979720,0.979720,0.979720,0.000000,28.067677,0.000000,89.199997,3.208449e-01,0.346782,2.418600,0.439069,legacy zeros may combine true zeros and unavailable-source zero-fill
1,i10fg,guarded matched points,not stored,m s-1,legacy product; inspect processing,see V2 data dictionary,137280,1.000000,1.000000,1.000000,1.000000,0.320647,20.863410,0.710000,44.720001,2.136122e+00,2.136122,2.739491,0.882870,legacy zeros may combine true zeros and unavailable-source zero-fill
2,wspd10,guarded matched points,not stored,m s-1,legacy product; inspect processing,see V2 data dictionary,137280,1.000000,1.000000,1.000000,1.000000,0.000470,16.529590,0.180000,28.590000,1.942067e+00,1.942067,2.430349,0.847758,legacy zeros may combine true zeros and unavailable-source zero-fill
3,t2m,guarded matched points,not stored,K,legacy product; inspect processing,see V2 data dictionary,137280,1.000000,1.000000,1.000000,1.000000,265.025460,313.157908,265.025452,313.157898,6.260659e-09,0.000008,0.000009,1.000000,legacy zeros may combine true zeros and unavailable-source zero-fill
4,EchoTop_18_00.50,guarded matched points,not stored,km,50-km mean; no-coverage bins zero-filled,see V2 data dictionary,137280,0.192818,0.902098,0.876537,0.124738,0.000000,16.475838,0.000000,19.000000,2.949047e+00,2.949047,5.369362,0.664585,legacy zeros may combine true zeros and unavailable-source zero-fill
5,MergedReflectivityComposite_00.50,guarded matched points,not stored,dBZ,50-km mean; no-coverage bins zero-filled,see V2 data dictionary,137280,0.941914,0.902098,0.876537,0.873805,-99.000000,48.892093,-30.000000,83.000000,7.513313e+01,76.631183,86.590369,0.293125,legacy zeros may combine true zeros and unavailable-source zero-fill


In [44]:
V2_CELL_STATISTIC = {
    "MultiSensor_QPE_01H_Pass2_00.00": "max of gauge-corrected 1-h accumulation over HH window; support fraction reported",
    "i10fg": "maximum of 2.5-km RTMA gust analyses within 50-km cell at HH:00",
    "wspd10": "maximum of 2.5-km RTMA wind-speed analyses within 50-km cell at HH:00",
    "t2m": "mean of 2.5-km RTMA temperature analyses within cell at HH:00",
    "t2m_max": "maximum of 2.5-km RTMA temperatures within cell at HH:00",
    "t2m_min": "minimum of 2.5-km RTMA temperatures within cell at HH:00",
    "NLDN_CG_015min_AvgDensity_00.00": "maximum 15-min average CG flash density during hour",
    "NLDN_CG_flashes_1h": "estimated flashes/cell/hour from hourly mean density x cell area",
    "MESH_Max_60min_00.50": "maximum estimated hail size (60-min track) within cell",
    "EchoTop_18_00.50": "maximum over six MRMS source files in hour",
    "MergedReflectivityComposite_00.50": "maximum over six MRMS source files in hour",
}
if not old_v2_comparison.empty:
    old_v2_comparison["v2_cell_statistic"] = old_v2_comparison.variable.map(V2_CELL_STATISTIC).fillna("see V2 data dictionary")
    display(old_v2_comparison[["variable", "old_cell_statistic", "v2_cell_statistic",
                              "old_valid_fraction_after_guards", "v2_valid_fraction",
                              "paired_supported_fraction", "old_max", "v2_max", "correlation"]])

,variable,old_cell_statistic,v2_cell_statistic,old_valid_fraction_after_guards,v2_valid_fraction,paired_supported_fraction,old_max,v2_max,correlation
0,MultiSensor_QPE_01H_Pass2_00.00,50-km mean; no-coverage bins zero-filled,max of gauge-corrected 1-h accumulation over HH window; support fraction reported,1.000000,0.979720,0.979720,28.067677,89.199997,0.439069
1,i10fg,legacy product; inspect processing,maximum of 2.5-km RTMA gust analyses within 50-km cell at HH:00,1.000000,1.000000,1.000000,20.863410,44.720001,0.882870
2,wspd10,legacy product; inspect processing,maximum of 2.5-km RTMA wind-speed analyses within 50-km cell at HH:00,1.000000,1.000000,1.000000,16.529590,28.590000,0.847758
3,t2m,legacy product; inspect processing,mean of 2.5-km RTMA temperature analyses within cell at HH:00,1.000000,1.000000,1.000000,313.157908,313.157898,1.000000
4,EchoTop_18_00.50,50-km mean; no-coverage bins zero-filled,maximum over six MRMS source files in hour,0.192818,0.902098,0.124738,16.475838,19.000000,0.664585
5,MergedReflectivityComposite_00.50,50-km mean; no-coverage bins zero-filled,maximum over six MRMS source files in hour,0.941914,0.902098,0.873805,48.892093,83.000000,0.293125


## 7. Implement V2 hazard-definition experiments

Starter candidates are deliberately explicit but **not approved thresholds**: 25 mm in the preceding one-hour QPE accumulation, 25 m/s gust, heat index/air-temperature maximum of 35 °C, and air-temperature minimum of −10 °C. RTMA is an analysis at HH:00 while MRMS fields represent the labeled hour/window; preserve this distinction. Missing pixels are excluded from denominators, not converted to zero. Spatial minimums and persistence are user-adjustable sensitivity parameters.

In [37]:
DEFINITION_CANDIDATES = pd.DataFrame([
    {"hazard": "extreme_precip", "variable": "MultiSensor_QPE_01H_Pass2_00.00", "operator": ">=", "threshold": 25.0, "units": "mm per 1-h accumulation", "threshold_status": "exploratory; verify gauge/source semantics"},
    {"hazard": "extreme_wind", "variable": "i10fg", "operator": ">=", "threshold": 25.0, "units": "m/s gust", "threshold_status": "exploratory; verify RTMA valid time and engineering criterion"},
    {"hazard": "extreme_heat", "variable": "t2m_max", "operator": ">=", "threshold": 308.15, "units": "K (35 C)", "threshold_status": "screening candidate; add duration and climatological context"},
    {"hazard": "extreme_cold", "variable": "t2m_min", "operator": "<=", "threshold": 263.15, "units": "K (-10 C)", "threshold_status": "screening candidate; add duration and climatological context"},
])

# Sensitivity dimensions, not universal physical constants.
MIN_ACTIVE_CELLS = 10
MIN_VALID_CELL_FRACTION = 0.50
MIN_PERSISTENCE_HOURS = {"extreme_precip": 1, "extreme_wind": 1, "extreme_heat": 3, "extreme_cold": 3}
SEVERITY_BINS = {
    "precip_mm": [0, 1, 10, 25, 50, np.inf],
    "gust_m_s": [0, 10, 20, 25, 33, np.inf],
    "temperature_C": [-np.inf, -20, -10, 0, 30, 35, 40, np.inf],
}


def load_v2_case(case_row, variables=None):
    start, end = case_row.start_time, case_row.end_time
    expected_days = pd.date_range(start.normalize(), end.normalize(), freq="D")
    path_by_day = {
        pd.Timestamp(pd.to_datetime(p.stem.rsplit("_", 1)[-1], format="%Y%m%d")).tz_localize("UTC").normalize(): p
        for p in V2_DIR.glob("weather_CONUS_*.nc")
    }
    paths = [path_by_day[day] for day in expected_days if day in path_by_day]
    if not paths:
        return None, {"expected_days": len(expected_days), "files_found": 0,
                      "missing_days": expected_days.astype(str).tolist()}
    parts = []
    for path in paths:
        with xr.open_dataset(path) as raw:
            keep = list(raw.data_vars) if variables is None else [v for v in variables if v in raw.data_vars]
            parts.append(raw[keep].load())
    dataset = xr.concat(parts, dim="time", data_vars="minimal", coords="minimal", compat="override").sortby("time")
    time = pd.DatetimeIndex(pd.to_datetime(dataset.time.values))
    if time.has_duplicates:
        raise ValueError(f"Duplicate V2 timestamps in {case_row.case_id}")
    dataset = dataset.sel(time=slice(start.tz_localize(None), end.tz_localize(None)))
    missing_days = expected_days.difference(pd.DatetimeIndex(list(path_by_day)))
    return dataset, {"expected_days": len(expected_days), "files_found": len(paths),
                     "missing_days": missing_days.astype(str).tolist(), "time_samples": dataset.sizes.get("time", 0)}


def field_coverage_report(dataset, case_id):
    rows = []
    for name, da in dataset.data_vars.items():
        values = np.asarray(da.values, dtype=float)
        finite = np.isfinite(values)
        rows.append({"case_id": case_id, "variable": name, "units": da.attrs.get("units"),
                     "n_total": values.size, "n_valid": int(finite.sum()),
                     "valid_fraction": float(finite.mean()) if values.size else np.nan,
                     "n_observed_zero": int(np.count_nonzero(values[finite] == 0)),
                     "minimum": float(np.nanmin(values)) if finite.any() else np.nan,
                     "maximum": float(np.nanmax(values)) if finite.any() else np.nan})
    return pd.DataFrame(rows)


def _longest_true_runs(flags, times):
    flags = np.asarray(flags, dtype=bool)
    runs = []
    start = None
    for i, flag in enumerate(flags):
        contiguous = i == 0 or times[i] - times[i - 1] == pd.Timedelta(hours=1)
        if flag and (start is None or not contiguous):
            if start is not None:
                runs.append((start, i - 1))
            start = i
        elif not flag and start is not None:
            runs.append((start, i - 1))
            start = None
    if start is not None:
        runs.append((start, len(flags) - 1))
    return runs


def detect_case_candidates(dataset, case_row, definitions=DEFINITION_CANDIDATES):
    records, hourly_rows = [], []
    if dataset is None:
        return pd.DataFrame(), pd.DataFrame()
    times = pd.DatetimeIndex(pd.to_datetime(dataset.time.values, utc=True))
    for definition in definitions.itertuples(index=False):
        if definition.variable not in dataset:
            continue
        values = np.asarray(dataset[definition.variable].values, dtype=float)
        if values.ndim != 3:
            continue
        valid = np.isfinite(values)
        if definition.operator == ">=":
            active = valid & (values >= definition.threshold)
        elif definition.operator == "<=":
            active = valid & (values <= definition.threshold)
        else:
            raise ValueError(f"Unknown operator {definition.operator}")
        n_valid = valid.sum(axis=(1, 2))
        n_active = active.sum(axis=(1, 2))
        fraction = np.divide(n_active, n_valid, out=np.full(len(times), np.nan), where=n_valid > 0)
        qualifying = (n_active >= MIN_ACTIVE_CELLS) & (fraction >= MIN_VALID_CELL_FRACTION)
        for i, timestamp in enumerate(times):
            hourly_rows.append({"case_id": case_row.case_id, "hazard": definition.hazard,
                                "time": timestamp, "variable": definition.variable,
                                "threshold": definition.threshold, "units": definition.units,
                                "n_valid_cells": int(n_valid[i]), "n_active_cells": int(n_active[i]),
                                "active_fraction_of_valid": fraction[i], "qualifying_hour": bool(qualifying[i])})
        for first, last in _longest_true_runs(qualifying, times):
            duration = last - first + 1
            if duration < MIN_PERSISTENCE_HOURS[definition.hazard]:
                continue
            records.append({"candidate_id": f"{case_row.case_id}_{definition.hazard}_{len(records)+1:03d}",
                            "case_id": case_row.case_id, "case_name": case_row.case_name,
                            "hazard": definition.hazard, "start_time": times[first], "end_time": times[last],
                            "duration_hours": duration, "peak_active_cells": int(n_active[first:last+1].max()),
                            "peak_active_fraction": float(np.nanmax(fraction[first:last+1])),
                            "threshold": definition.threshold, "units": definition.units,
                            "definition_status": definition.threshold_status,
                            "reference_status": case_row.validation_status})
    return pd.DataFrame(records), pd.DataFrame(hourly_rows)


print(DEFINITION_CANDIDATES.to_string(index=False))

        hazard                        variable operator  threshold                   units                                              threshold_status
extreme_precip MultiSensor_QPE_01H_Pass2_00.00       >=      25.00 mm per 1-h accumulation                    exploratory; verify gauge/source semantics
  extreme_wind                           i10fg       >=      25.00                m/s gust exploratory; verify RTMA valid time and engineering criterion
  extreme_heat                         t2m_max       >=     308.15                K (35 C)  screening candidate; add duration and climatological context
  extreme_cold                         t2m_min       <=     263.15               K (-10 C)  screening candidate; add duration and climatological context


In [14]:
# Bounded loader smoke test: one day, three physical fields, no event scoring.
SMOKE_TEST_DATE = pd.Timestamp("2024-05-10", tz="UTC")
smoke_case = CASE_REGISTRY.loc[CASE_REGISTRY.case_name.eq("Gannon storm candidate")].iloc[0].copy()
smoke_case["start_time"] = SMOKE_TEST_DATE
smoke_case["end_time"] = SMOKE_TEST_DATE + pd.Timedelta(hours=23)
smoke_case["case_id"] = "v2_loader_smoke_20240510"
smoke_dataset, smoke_report = load_v2_case(
    smoke_case, variables=["MultiSensor_QPE_01H_Pass2_00.00", "i10fg", "t2m"]
)
assert smoke_dataset is not None, "Expected the staged V2 May 10 file to load."
assert smoke_dataset.sizes.get("time") == 24, smoke_report
assert {"MultiSensor_QPE_01H_Pass2_00.00", "i10fg", "t2m"}.issubset(smoke_dataset.data_vars)
smoke_coverage = field_coverage_report(smoke_dataset, smoke_case.case_id)
print("One-day V2 loader smoke test passed:", smoke_report)
display(smoke_coverage)

One-day V2 loader smoke test passed: {'expected_days': 1, 'files_found': 1, 'missing_days': [], 'time_samples': 24}


,case_id,variable,units,n_total,n_valid,valid_fraction,n_observed_zero,minimum,maximum
0,v2_loader_smoke_20240510,MultiSensor_QPE_01H_Pass2_00.00,mm,137280,134496,0.97972,120981,0.000000,89.199997
1,v2_loader_smoke_20240510,i10fg,m s-1,137280,137280,1.00000,0,0.710000,44.720001
2,v2_loader_smoke_20240510,t2m,K,137280,137280,1.00000,0,265.025452,313.157898


## 8. Compare V2 candidate outputs against known events

Matching uses a declared timing tolerance and, when both sources provide county FIPS, requires the same county. Event names/date windows alone are not verified references; the default registry is candidate-only and must not yield precision/recall. Confirmed scores require a reference file with `validation_status` set to `verified` or `confirmed` and a traceable source.

In [9]:
TIMING_TOLERANCE = pd.Timedelta(hours=3)


def compare_candidates_to_references(candidates, references, tolerance=TIMING_TOLERANCE):
    verified = references.loc[
        references.validation_status.astype(str).str.lower().isin({"verified", "confirmed"})
    ].copy() if not references.empty else pd.DataFrame()
    if verified.empty:
        return pd.DataFrame(), {"precision": np.nan, "recall": np.nan, "hits": 0,
                                "false_positives": np.nan, "misses": np.nan,
                                "status": "no_verified_reference_labels"}
    if candidates.empty:
        return pd.DataFrame(), {"precision": 0.0, "recall": 0.0, "hits": 0,
                                "false_positives": 0, "misses": len(verified),
                                "verified_reference_count": len(verified), "candidate_count": 0}

    possible_pairs = []
    for detected in candidates.itertuples(index=False):
        for reference in verified.itertuples(index=False):
            if str(detected.hazard).lower() != str(reference.hazard).lower():
                continue
            if not (detected.start_time <= reference.end_time + tolerance and
                    reference.start_time <= detected.end_time + tolerance):
                continue
            spatial_status = "temporal_only; geography unavailable"
            detected_fips = getattr(detected, "county_fips", None)
            reference_fips = getattr(reference, "county_fips", None)
            if detected_fips is not None and reference_fips is not None and pd.notna(detected_fips) and pd.notna(reference_fips):
                if str(detected_fips).zfill(5) != str(reference_fips).zfill(5):
                    continue
                spatial_status = "same_county"
            offset = (detected.start_time - reference.start_time).total_seconds() / 3600
            possible_pairs.append({"candidate_id": detected.candidate_id, "reference_id": reference.event_id,
                                   "hazard": detected.hazard, "temporal_offset_hours": offset,
                                   "absolute_start_offset_hours": abs(offset),
                                   "spatial_match_status": spatial_status, "reference_source": reference.source})

    # Greedy one-to-one assignment, closest onset first: one detection cannot score
    # several references, and a broad reference cannot validate many duplicate alarms.
    possible_pairs.sort(key=lambda pair: pair["absolute_start_offset_hours"])
    accepted, used_candidates, used_references = [], set(), set()
    for pair in possible_pairs:
        if pair["candidate_id"] in used_candidates or pair["reference_id"] in used_references:
            continue
        accepted.append(pair)
        used_candidates.add(pair["candidate_id"])
        used_references.add(pair["reference_id"])
    matched = pd.DataFrame(accepted).drop(columns=["absolute_start_offset_hours"], errors="ignore")
    tp = len(accepted)
    fp = len(candidates) - tp
    fn = len(verified) - tp
    metrics = {"precision": tp / (tp + fp) if tp + fp else np.nan,
               "recall": tp / (tp + fn) if tp + fn else np.nan,
               "hits": tp, "false_positives": fp, "misses": fn,
               "verified_reference_count": len(verified), "candidate_count": len(candidates),
               "matching": "greedy one-to-one by closest onset within hazard/tolerance/geography rules"}
    return matched, metrics


# Candidate event table only if the bounded loader and definition experiment are enabled.
if RUN_LOAD_CASES and RUN_DEFINITION_EXPERIMENTS:
    selected_case_rows = CASE_REGISTRY.loc[CASE_REGISTRY.case_name.isin(CASE_NAMES)]
    definitions = DEFINITION_CANDIDATES.copy()
    case_event_tables, case_hourly_tables, case_load_rows, case_coverage = [], [], [], []
    required_variables = definitions.variable.unique().tolist()
    for case_row in selected_case_rows.itertuples(index=False):
        ds_case, load_info = load_v2_case(case_row, variables=required_variables)
        case_load_rows.append({"case_id": case_row.case_id, **load_info})
        if ds_case is None:
            continue
        case_coverage.append(field_coverage_report(ds_case, case_row.case_id))
        events, hourly = detect_case_candidates(ds_case, case_row, definitions)
        case_event_tables.append(events)
        case_hourly_tables.append(hourly)
    candidate_events = pd.concat(case_event_tables, ignore_index=True) if case_event_tables else pd.DataFrame()
    hourly_detections = pd.concat(case_hourly_tables, ignore_index=True) if case_hourly_tables else pd.DataFrame()
    case_load_report = pd.DataFrame(case_load_rows)
    field_coverage = pd.concat(case_coverage, ignore_index=True) if case_coverage else pd.DataFrame()
    print("Candidate events are not validated events. Review coverage before metrics.")
    display(case_load_report)
    display(candidate_events)
else:
    candidate_events = pd.DataFrame()
    hourly_detections = pd.DataFrame()
    case_load_report = pd.DataFrame()
    field_coverage = pd.DataFrame()
    print("V2 case load / hazard experiment skipped. Set RUN_LOAD_CASES and RUN_DEFINITION_EXPERIMENTS after reviewing case coverage.")

matched_pairs, performance = compare_candidates_to_references(candidate_events, known_events)
print("Reference metrics:", performance)

V2 case load / hazard experiment skipped. Set RUN_LOAD_CASES and RUN_DEFINITION_EXPERIMENTS after reviewing case coverage.
Reference metrics: {'precision': nan, 'recall': nan, 'hits': 0, 'false_positives': nan, 'misses': nan, 'status': 'no_verified_reference_labels'}


In [15]:
# Synthetic metric check: one-to-one matching must not let duplicate detections inflate recall.
_test_candidates = pd.DataFrame([
    {"candidate_id": "det-1", "hazard": "extreme_precip", "start_time": pd.Timestamp("2024-05-10T01:00Z"), "end_time": pd.Timestamp("2024-05-10T02:00Z"), "county_fips": "06037"},
    {"candidate_id": "det-2", "hazard": "extreme_precip", "start_time": pd.Timestamp("2024-05-10T01:30Z"), "end_time": pd.Timestamp("2024-05-10T02:30Z"), "county_fips": "06037"},
])
_test_references = pd.DataFrame([
    {"event_id": "ref-1", "hazard": "extreme_precip", "start_time": pd.Timestamp("2024-05-10T00:30Z"), "end_time": pd.Timestamp("2024-05-10T03:00Z"), "county_fips": "06037", "validation_status": "verified", "source": "synthetic test"},
])
_test_pairs, _test_metrics = compare_candidates_to_references(_test_candidates, _test_references)
assert len(_test_pairs) == 1 and _test_metrics["hits"] == 1
assert _test_metrics["false_positives"] == 1 and _test_metrics["misses"] == 0
print("Synthetic event-matching check passed:", _test_metrics)

Synthetic event-matching check passed: {'precision': 0.5, 'recall': 1.0, 'hits': 1, 'false_positives': 1, 'misses': 0, 'verified_reference_count': 1, 'candidate_count': 2, 'matching': 'greedy one-to-one by closest onset within hazard/tolerance/geography rules'}


## 9. Summarize mismatches and export reproducible artifacts

Exports are written to a new notebook output folder. The notebook keeps source values and missingness intact; only diagnostics and candidate tables are exported. If verified event labels are absent, it will report candidate detections and coverage but will not manufacture performance scores.

In [46]:
RUN_OUTPUT_DIR = EVENT_ARTIFACT_DIR / pd.Timestamp.utcnow().strftime("run_%Y%m%dT%H%M%SZ")
print("This run will write to a new directory and will not overwrite prior artifacts:", RUN_OUTPUT_DIR)

This run will write to a new directory and will not overwrite prior artifacts: /Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation/run_20261029T163125Z


In [47]:
verified_references = known_events.loc[
    known_events.validation_status.astype(str).str.lower().isin({"verified", "confirmed"})
].copy() if not known_events.empty else pd.DataFrame()
if not candidate_events.empty and not verified_references.empty:
    matched_candidate_ids = set(matched_pairs.candidate_id) if not matched_pairs.empty else set()
    matched_reference_ids = set(matched_pairs.reference_id) if not matched_pairs.empty else set()
    false_positive_candidates = candidate_events.loc[~candidate_events.candidate_id.isin(matched_candidate_ids)].copy()
    missed_reference_events = verified_references.loc[~verified_references.event_id.isin(matched_reference_ids)].copy()
else:
    false_positive_candidates = pd.DataFrame()
    missed_reference_events = pd.DataFrame()

# Plot hourly exceedance footprint for selected cases; each hazard remains a separate panel.
if not hourly_detections.empty:
    plot_hazards = hourly_detections.hazard.unique().tolist()
    fig, axes = plt.subplots(len(plot_hazards), 1, figsize=(13, max(3, 2.8 * len(plot_hazards))), sharex=True, squeeze=False)
    for ax, hazard in zip(axes[:, 0], plot_hazards):
        subset = hourly_detections.loc[hourly_detections.hazard.eq(hazard)]
        for case_id, group in subset.groupby("case_id"):
            ax.plot(group.time, group.active_fraction_of_valid, label=case_id, linewidth=1.2)
        ax.axhline(MIN_VALID_CELL_FRACTION, color="grey", linestyle="--", linewidth=0.8,
                   label="minimum valid-cell active fraction")
        ax.set_ylabel("Active / valid")
        ax.set_title(hazard)
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, ncol=3)
    axes[-1, 0].set_xlabel("UTC hour")
    fig.suptitle("V2 candidate exceedance footprints — exploratory thresholds")
    fig.tight_layout()
    plt.show()
else:
    print("No candidate time series to plot; enable the bounded load/experiment flags first.")

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
export_tables = {
    "case_registry.csv": CASE_REGISTRY,
    "validation_source_inventory.csv": validation_inventory,
    "pipeline_validation_inventory.csv": pipeline_validation_inventory,
    "hazard_definition_candidates.csv": DEFINITION_CANDIDATES,
    "case_load_report.csv": case_load_report,
    "v2_field_coverage.csv": field_coverage,
    "candidate_hazard_events.csv": candidate_events,
    "hourly_candidate_detections.csv": hourly_detections,
    "matched_candidate_reference_pairs.csv": matched_pairs,
    "candidate_false_positives.csv": false_positive_candidates,
    "missed_verified_references.csv": missed_reference_events,
}
for filename, table in export_tables.items():
    if table is not None:
        table.to_csv(RUN_OUTPUT_DIR / filename, index=False)

comparison_manifest = {
    "pipeline_notebook": str(PIPELINE_NOTEBOOK),
    "v2_data_root": str(V2_DIR),
    "data_dictionary": str(DATA_DICTIONARY),
    "case_names_selected": CASE_NAMES,
    "case_file_coverage": CASE_REGISTRY[["case_id", "v2_days_present", "expected_days"]].to_dict(orient="records"),
    "thresholds_status": "exploratory examples; not production definitions",
    "metrics": performance,
    "known_event_reference": str(KNOWN_EVENTS_FILE) if KNOWN_EVENTS_FILE else None,
    "verified_reference_count": len(verified_references),
    "time_tolerance": str(TIMING_TOLERANCE),
    "missingness_policy": "NaN remains no coverage; zero remains observed zero",
    "limitations": ["candidate windows are not ground-truth labels",
                     "case coverage is incomplete for several 2024 windows",
                     "event candidates are grid/time summaries, not final connected spatial objects"],
}
(RUN_OUTPUT_DIR / "manifest.json").write_text(json.dumps(comparison_manifest, indent=2, default=str))
print("Saved reproducible review artifacts to", RUN_OUTPUT_DIR)
print("No verified references loaded:" if verified_references.empty else "Verified references:", len(verified_references))

No candidate time series to plot; enable the bounded load/experiment flags first.
Saved reproducible review artifacts to /Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation/run_20261029T163125Z
No verified references loaded: 0


In [48]:
if not old_v2_comparison.empty:
    old_v2_comparison.to_csv(RUN_OUTPUT_DIR / "old_v2_matched_day_comparison.csv", index=False)
    (RUN_OUTPUT_DIR / "old_v2_processing_semantics.md").write_text(
        "Legacy MRMS gridding used 50-km means and converted empty bins to zero. "
        "V2 MRMS uses documented cell maxima and valid-pixel fractions. RTMA V2 values "
        "are analyses valid at HH:00, not hourly means. Legacy files lack variable unit "
        "and source-coverage metadata. Old negative sentinels are masked with field-specific "
        "guards, but legacy zero-fill cannot be reliably distinguished from observed zero. "
        "Therefore summary differences are diagnostic, not accuracy scores or proof that V2 fixes the source.\n"
    )
    print("Saved guarded old/V2 comparison and semantics to", RUN_OUTPUT_DIR)

Saved guarded old/V2 comparison and semantics to /Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation/run_20261029T163125Z


### Interpretation and next steps

1. Review the repository/pipeline inventory first; execution counts in stored notebook JSON are historical and do not prove the current source cell passes.
2. Select a V2-covered case. Start with Northeast Maine heat (complete V2 files) or the May 10–18 portion of Gannon; the latter is a storm context case, not a terrestrial-weather truth label.
3. Confirm independent event boundaries and location before marking any event reference `verified`. Add reference provenance and geography to the event table.
4. Review matched old/V2 differences at source support and units before interpreting candidate counts. Then vary one hazard-specific definition dimension at a time (threshold, duration, spatial footprint/support).
5. Export candidate detections and false-positive/miss tables only after the reference labels are verified. Preserve candidate thresholds as exploratory until case review and sensitivity checks support a decision.

This notebook intentionally does not rewrite MYRIAD definitions, rebuild full-period caches, or use Eagle-I/GIC outcomes as ground truth for terrestrial-weather hazards.

## 10. Three-panel V2 hazard movie with candidate boundaries

Render a bounded, 72-hour May 10–12, 2024 example from V2: hourly gauge-corrected precipitation on the left, gust in the middle, and maximum 2 m temperature on the right. County/state boundaries are shown for geographic reference; red outlines mark cells exceeding the explicitly exploratory candidate thresholds above. They are **instantaneous grid-cell threshold boundaries**, not verified storm/event perimeters. NaNs stay transparent and are excluded from coverage; observed zero precipitation remains visible. This does not read or modify the MH database pipeline outputs.

In [55]:
# Movie controls. Use the complete V2 subset of the Gannon-storm candidate window.
MOVIE_START = pd.Timestamp("2024-05-10 00:00:00", tz="UTC")
MOVIE_END = pd.Timestamp("2024-05-12 23:00:00", tz="UTC")
MOVIE_FREQ = "1h"
MOVIE_FPS = 4
MOVIE_DPI = 120
MOVIE_MIN_VALID_FRACTION = 0.80
MOVIE_MIN_ACTIVE_CELLS = 1
MOVIE_OVERWRITE = True  # Preserve the input coverage mask while re-rendering with non-overlapping axes.
MOVIE_PATH = EVENT_ARTIFACT_DIR / "v2_terrestrial_hazards_20240510_20240512.mp4"
STATE_BOUNDARY_PATH = REPO_ROOT / "data" / "cb_2023_us_state_20m.geojson"

MOVIE_FIELDS = {
    "precip": {"variable": "MultiSensor_QPE_01H_Pass2_00.00", "threshold": 25.0,
               "units": "mm / 1 h", "cmap": "Blues", "label": "1-h gauge-corrected precipitation"},
    "wind": {"variable": "i10fg", "threshold": 25.0,
              "units": "m/s gust", "cmap": "YlOrRd", "label": "10-m gust"},
    "temperature": {"variable": "t2m_max", "threshold": 308.15,
                     "units": "K (35 C)", "cmap": "magma", "label": "2-m temperature maximum at HH analysis"},
}


def _load_v2_movie_fields(start, end):
    expected_days = pd.date_range(start.normalize(), end.normalize(), freq="D")
    path_by_day = {
        pd.Timestamp(pd.to_datetime(path.stem.rsplit("_", 1)[-1], format="%Y%m%d")).tz_localize("UTC").normalize(): path
        for path in V2_DIR.glob("weather_CONUS_*.nc")
    }
    missing = expected_days.difference(pd.DatetimeIndex(list(path_by_day)))
    if len(missing):
        raise FileNotFoundError(f"V2 movie period has missing daily files: {missing.astype(str).tolist()}")
    variables = [item["variable"] for item in MOVIE_FIELDS.values()]
    coverage_variables = [f"{name}_valid_frac" for name in variables] + ["rtma_valid_frac"]
    parts = []
    for day in expected_days:
        path = path_by_day[day]
        with xr.open_dataset(path) as raw:
            absent = [name for name in variables if name not in raw]
            if absent:
                raise KeyError(f"{path.name} lacks movie fields {absent}")
            keep = variables + [name for name in coverage_variables if name in raw]
            keep.extend(name for name in ["lat", "lon", "x", "y"] if name in raw and name not in keep)
            parts.append(raw[keep].load())
    dataset = xr.concat(parts, dim="time", data_vars="minimal", coords="minimal", compat="override").sortby("time")
    times = pd.DatetimeIndex(pd.to_datetime(dataset.time.values, utc=True))
    if times.has_duplicates:
        raise ValueError("Duplicate time labels in V2 movie inputs.")
    dataset = dataset.sel(time=slice(start.tz_localize(None), end.tz_localize(None)))
    expected = pd.date_range(start, end, freq=MOVIE_FREQ)
    actual = pd.DatetimeIndex(pd.to_datetime(dataset.time.values, utc=True))
    if not actual.equals(expected):
        raise ValueError(f"Expected complete hourly frame index; got {len(actual)} of {len(expected)} labels.")
    return dataset, expected


def _coverage_for_movie_field(dataset, variable):
    coverage_name = f"{variable}_valid_frac"
    if coverage_name in dataset:
        return np.asarray(dataset[coverage_name].values, dtype=float)
    if variable in {"i10fg", "t2m_max"} and "rtma_valid_frac" in dataset:
        return np.asarray(dataset["rtma_valid_frac"].values, dtype=float)
    return np.isfinite(np.asarray(dataset[variable].values, dtype=float)).astype(float)


def render_v2_hazard_movie():
    import os
    import shutil
    from matplotlib.animation import FFMpegWriter, FuncAnimation
    from IPython.display import Video, display

    if MOVIE_PATH.exists() and not MOVIE_OVERWRITE:
        print("Existing movie preserved; displaying it:", MOVIE_PATH)
        display(Video(str(MOVIE_PATH), embed=False, html_attributes="controls"))
        return
    if not STATE_BOUNDARY_PATH.exists():
        raise FileNotFoundError(STATE_BOUNDARY_PATH)
    if not shutil.which("ffmpeg"):
        try:
            import imageio_ffmpeg
            plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
        except ImportError as exc:
            raise RuntimeError("ffmpeg is required to export this movie.") from exc

    import geopandas as gpd
    states = gpd.read_file(STATE_BOUNDARY_PATH)
    if states.crs is None:
        states = states.set_crs("EPSG:4326")
    states = states.to_crs("EPSG:5070")
    if "STUSPS" in states.columns:
        states = states.loc[~states.STUSPS.isin(["AK", "HI", "PR", "VI", "GU", "MP", "AS", "UM"])].copy()
    states_km = states.copy()
    states_km["geometry"] = states.geometry.scale(xfact=0.001, yfact=0.001, origin=(0, 0))

    dataset, frame_times = _load_v2_movie_fields(MOVIE_START, MOVIE_END)
    if "x" not in dataset.coords or "y" not in dataset.coords:
        raise KeyError("V2 data must provide projected x/y coordinates for accurate cell plotting.")
    x_centers, y_centers = np.asarray(dataset.x.values, dtype=float), np.asarray(dataset.y.values, dtype=float)
    if not (np.all(np.diff(x_centers) > 0) and np.all(np.diff(y_centers) > 0)):
        raise ValueError("Projected x/y cell centers must be strictly increasing.")
    dx, dy = np.median(np.diff(x_centers)), np.median(np.diff(y_centers))
    x_edges = np.r_[x_centers[0] - dx / 2, x_centers + dx / 2]
    y_edges = np.r_[y_centers[0] - dy / 2, y_centers + dy / 2]

    fields, valid_masks, candidate_masks, limits, diagnostics = {}, {}, {}, {}, []
    for key, spec in MOVIE_FIELDS.items():
        values = np.asarray(dataset[spec["variable"]].values, dtype=float)
        support = _coverage_for_movie_field(dataset, spec["variable"])
        if support.shape != values.shape:
            support = np.broadcast_to(support, values.shape)
        valid = np.isfinite(values) & np.isfinite(support) & (support >= MOVIE_MIN_VALID_FRACTION)
        active = valid & (values >= spec["threshold"])
        fields[key] = np.where(valid, values, np.nan)
        valid_masks[key] = valid
        candidate_masks[key] = active
        finite_values = values[valid]
        if not len(finite_values):
            raise ValueError(f"No values meet the support gate for {spec['variable']}.")
        upper = float(np.nanpercentile(finite_values, 99.5))
        limits[key] = (float(np.nanpercentile(finite_values, 1)), upper if upper > 0 else float(finite_values.max()))
        for index, timestamp in enumerate(frame_times):
            diagnostics.append({"time_utc": timestamp.isoformat(), "field": key,
                                "variable": spec["variable"], "valid_cells": int(valid[index].sum()),
                                "supported_domain_cells": int(np.isfinite(support[index]).sum()),
                                "active_cells": int(active[index].sum()), "threshold": spec["threshold"],
                                "units": spec["units"], "threshold_exceedance_exists": bool(active[index].any())})

    fig, axes = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=False)
    fig.subplots_adjust(left=0.045, right=0.965, bottom=0.10, top=0.82, wspace=0.30)
    meshes, contours = {}, {key: None for key in MOVIE_FIELDS}
    for ax, (key, spec) in zip(axes, MOVIE_FIELDS.items()):
        vmin, vmax = limits[key]
        if key == "precip":
            vmin = 0.0
        mesh = ax.pcolormesh(x_edges / 1000, y_edges / 1000,
                             np.ma.masked_invalid(fields[key][0]), shading="flat",
                             cmap=spec["cmap"], vmin=vmin, vmax=vmax,
                             rasterized=True, zorder=1)
        meshes[key] = mesh
        states_km.boundary.plot(ax=ax, color="#333333", linewidth=0.55, alpha=0.85, zorder=3)
        ax.set_xlim(x_edges[0] / 1000, x_edges[-1] / 1000)
        ax.set_ylim(y_edges[0] / 1000, y_edges[-1] / 1000)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("EPSG:5070 easting (km)")
        ax.grid(alpha=0.12, zorder=0)
        ax.set_title(spec["label"], fontsize=11, pad=12)
        fig.colorbar(mesh, ax=ax, shrink=0.76, pad=0.025, label=spec["units"])
    axes[0].set_ylabel("EPSG:5070 northing (km)")
    suptitle = fig.suptitle("", fontsize=14, y=0.97)

    # Include the full grid plus all of CONUS; retain the dataset's existing coverage masks.
    grid_bounds = (x_edges[0] / 1000, y_edges[0] / 1000,
                   x_edges[-1] / 1000, y_edges[-1] / 1000)
    states_bounds = states_km.total_bounds
    map_bounds = (min(grid_bounds[0], states_bounds[0]), min(grid_bounds[1], states_bounds[1]),
                  max(grid_bounds[2], states_bounds[2]), max(grid_bounds[3], states_bounds[3]))
    for ax in axes:
        ax.set_xlim(map_bounds[0], map_bounds[2])
        ax.set_ylim(map_bounds[1], map_bounds[3])

    def update(frame_index):
        timestamp = frame_times[frame_index]
        for axis_index, (key, spec) in enumerate(MOVIE_FIELDS.items()):
            values = fields[key][frame_index]
            active = candidate_masks[key][frame_index]
            meshes[key].set_array(np.ma.masked_invalid(values).ravel())
            if contours[key] is not None:
                contours[key].remove()
                contours[key] = None
            if int(active.sum()) >= MOVIE_MIN_ACTIVE_CELLS:
                contours[key] = axes[axis_index].contour(
                    x_centers / 1000, y_centers / 1000, active.astype(float), levels=[0.5],
                    colors=["red"], linewidths=1.15, corner_mask=False, zorder=5)
            coverage = float(valid_masks[key][frame_index].mean())
            axes[axis_index].set_title(
                f"{spec['label']}\nthreshold {spec['threshold']:g} {spec['units']} | "
                f"active cells={int(active.sum())} | valid={coverage:.0%}", fontsize=10, pad=12)
        suptitle.set_text(f"V2 terrestrial hazards | {timestamp:%Y-%m-%d %H:%M UTC}\n"
                          "Red contours = exploratory per-hour exceedances; state lines for reference")
        return tuple(meshes.values())

    animation = FuncAnimation(fig, update, frames=len(frame_times), interval=1000 / MOVIE_FPS,
                              blit=False, repeat=False, cache_frame_data=False)
    MOVIE_PATH.parent.mkdir(parents=True, exist_ok=True)
    partial = MOVIE_PATH.with_name(MOVIE_PATH.stem + ".partial.mp4")
    if partial.exists():
        raise FileExistsError(f"Partial movie exists; inspect before removal: {partial}")
    writer = FFMpegWriter(fps=MOVIE_FPS, codec="libx264", bitrate=2200,
                          metadata={"title": "V2 precipitation, wind gust and temperature"},
                          extra_args=["-pix_fmt", "yuv420p", "-movflags", "+faststart",
                                      "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2"])
    try:
        animation.save(str(partial), writer=writer, dpi=MOVIE_DPI)
        if MOVIE_PATH.exists() and not MOVIE_OVERWRITE:
            raise FileExistsError(MOVIE_PATH)
        os.replace(partial, MOVIE_PATH)
    finally:
        plt.close(fig)
        if partial.exists():
            partial.unlink()

    summary = pd.DataFrame(diagnostics)
    summary_path = MOVIE_PATH.with_suffix(".csv")
    summary.to_csv(summary_path, index=False)
    print({"movie": str(MOVIE_PATH), "frames": len(frame_times), "fps": MOVIE_FPS,
           "thresholds": {key: value["threshold"] for key, value in MOVIE_FIELDS.items()},
           "minimum_valid_fraction": MOVIE_MIN_VALID_FRACTION,
           "candidate_cell_hours": summary.groupby("field").active_cells.sum().to_dict(),
           "summary": str(summary_path),
           "caveat": "Exploratory grid-cell threshold boundaries, not independently validated event perimeters."})
    display(Video(str(MOVIE_PATH), embed=False, html_attributes="controls"))


RUN_MOVIE = True
if RUN_MOVIE:
    render_v2_hazard_movie()
else:
    print("Set RUN_MOVIE=True to export the bounded 72-frame V2 movie.")

{'movie': '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation/v2_terrestrial_hazards_20240510_20240512.mp4', 'frames': 72, 'fps': 4, 'thresholds': {'precip': 25.0, 'wind': 25.0, 'temperature': 308.15}, 'minimum_valid_fraction': 0.8, 'candidate_cell_hours': {'precip': 903, 'temperature': 950, 'wind': 100}, 'summary': '/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/notebooks/outputs/terrestrial_v2_validation/v2_terrestrial_hazards_20240510_20240512.csv', 'caveat': 'Exploratory grid-cell threshold boundaries, not independently validated event perimeters.'}


## Pause / resume checkpoint (2026-10-29)

**Current state:** The standalone V2 validation notebook and its 72-frame, 4-fps three-panel movie have been rendered successfully for May 10–12, 2024. The movie uses projected EPSG:5070 coordinates, state boundaries, an 80% valid-support display gate, and exploratory exceedance contours: QPE ≥25 mm/1 h, gust ≥25 m/s, and `t2m_max` ≥308.15 K. The adjacent CSV contains hourly coverage and active-cell summaries. These contours are candidate grid-cell boundaries, not independently verified event perimeters.

**Resume here:** Visually inspect the most recently extracted first frame at `/tmp/v2_hazard_movie_first_frame.png` to finish checking the final layout/alignment. If it looks good, change `MOVIE_OVERWRITE` to `False` in the movie cell before rerunning anything, to protect the completed MP4. Then run only that movie cell if needed—not the whole notebook. The notebook's old/V2 diagnostics and smoke tests already ran; no independent known-event truth references are loaded (`KNOWN_EVENTS_FILE = None`), so no validation precision/recall claims have been made.

**Artifacts:** `notebooks/outputs/terrestrial_v2_validation/v2_terrestrial_hazards_20240510_20240512.mp4` and its matching `.csv`.